In [1]:
# Prototype for dashboard computations

# Performance
# ytd perf

# Risk
# 20-day historical vol
# 1-day VaR

# Just storing stuff in the DB. Only portfolio level for now
# First compute portfolio values

In [48]:
from worker.database import connect

import polars as pl
import datetime

In [97]:
DATE = datetime.date(2025, 8, 29)
DATE_CALC_START = datetime.date(2023, 9, 1)
DATE_START = datetime.date(2018, 9, 1)

with connect() as session:
    df_positions = pl.read_database(
        """
        SELECT
            portfolio_id::TEXT as portfolio_id,
            date,
            instrument_id::TEXT as instrument_id,
            quantity
        FROM ptf_comp
        WHERE date <= :date_end
        AND date >= :date_start
    """,
        session,
        execute_options={"params": {"date_end": DATE, "date_start": DATE_CALC_START}},
    )

    u_insts = df_positions["instrument_id"].unique().to_list()

    df_market_data = pl.read_database(
        """
        SELECT
            instrument_id::TEXT as instrument_id,
            date,
            value as price
        FROM market_data
        WHERE 1=1
        AND date >= :date_start
        AND date <= :date_end
        AND instrument_id::TEXT = ANY(:insts)
        AND data_type = 'adj_close'
    """,
        session,
        execute_options={"params": {"date_start": DATE_START, "date_end": DATE, "insts": u_insts}},
    )

In [98]:
df_positions

portfolio_id,date,instrument_id,quantity
str,date,str,"decimal[*,8]"
"""e3a50d76-dcb7-4377-8ad3-8ff9f4…",2025-08-29,"""fb39c249-6cb1-47b6-9220-6c0f8d…",1000.00000000
"""4daf79fe-feb9-4c5a-ba1f-a67b96…",2025-08-29,"""6c125ed4-4a90-41c4-b389-226eb7…",1000.00000000
"""d7da851b-20c5-4de0-9c93-68a9f1…",2025-08-29,"""02f2fb63-27d1-43bf-953d-7249c4…",1000.00000000
"""0b209d5c-12ed-48af-a597-b1e22f…",2025-08-29,"""41712141-41c4-4e22-83d1-8cc45c…",1000.00000000
"""f4dd8467-36cf-4d10-93d3-a0699a…",2025-08-29,"""c135cb6f-2a23-4032-ae80-856a31…",1000.00000000
…,…,…,…
"""74623cfa-69a7-43c4-a6e5-3beb9b…",2025-08-26,"""35a6016b-daf4-40a4-84d1-c5492a…",1000.00000000
"""73b9f260-1100-469b-99a2-3a5f9e…",2025-08-26,"""56cfb131-8c13-4e90-a0e9-55f93e…",1000.00000000
"""67756108-d9ee-44e0-ac13-0e133d…",2025-08-26,"""07602a17-b596-47d1-adc2-bda6d3…",1000.00000000


In [88]:
def with_returns(df: pl.DataFrame, date_col: str, value_col: str) -> pl.DataFrame:
    return df.sort(date_col).with_columns(returns=pl.col(value_col).pct_change()).drop_nulls()


def with_vols(df: pl.DataFrame, date_col: str, returns_col: str) -> pl.DataFrame:
    return (
        df.sort(date_col)
        .with_row_index(name="idx")
        .with_columns(vol=pl.col(returns_col).rolling("idx", period="20i"))
        .with_columns(vol=pl.when(pl.col("vol").list.len() == 20).then(pl.col("vol").list.std()).otherwise(None))
        .drop("idx")
    )


df_exante_perf = (
    df_positions.join(df_market_data, on="instrument_id")
    .with_columns(value=pl.col("quantity") * pl.col("price"))
    .group_by("portfolio_id", "date")
    .agg(value=pl.col("value").sum())
    .group_by("portfolio_id")
    .map_groups(lambda df: with_returns(df, date_col="date", value_col="value"))
)

In [ ]:
df_rolling_vols = df_exante_perf.group_by("portfolio_id").map_groups(
    lambda df: with_vols(df, date_col="date", returns_col="returns")
)

df_rolling_vols

portfolio_id,date,value,returns,vol
str,date,"decimal[*,20]",f64,f64
"""b872c12a-2db2-4898-a52d-4f09e8…",2018-09-05,58818.47382130000000000000,-0.005471,null
"""b872c12a-2db2-4898-a52d-4f09e8…",2018-09-06,58599.60553170000000000000,-0.003721,null
"""b872c12a-2db2-4898-a52d-4f09e8…",2018-09-07,57962.03268810000000000000,-0.01088,null
"""b872c12a-2db2-4898-a52d-4f09e8…",2018-09-10,58380.73724210000000000000,0.007224,null
"""b872c12a-2db2-4898-a52d-4f09e8…",2018-09-11,58266.54509100000000000000,-0.001956,null
…,…,…,…,…
"""f5764652-eca7-48f2-b621-b5fdff…",2025-08-22,62990.00000000000000000000,0.011725,0.004533
"""f5764652-eca7-48f2-b621-b5fdff…",2025-08-25,62720.00000000000000000000,-0.004286,0.004539
"""f5764652-eca7-48f2-b621-b5fdff…",2025-08-26,62850.00000000000000000000,0.002073,0.004547


In [94]:
def with_historical_vars(df: pl.DataFrame, n: int, cf: float, returns_col: str) -> pl.DataFrame:
    return (
        df.with_row_index(name="idx")
        .with_columns(rolled_returns=pl.col(returns_col).rolling("idx", period=f"{n}i"))
        .with_columns(
            var=pl.when(pl.col("rolled_returns").list.len() == n)
            .then(pl.col("rolled_returns").list.eval(pl.element().quantile(1 - cf)).list.get(0))
            .otherwise(None)
        )
    )


df_rolling_historical_var = df_rolling_vols.group_by("portfolio_id").map_groups(
    lambda df: with_historical_vars(df, n=500, cf=0.99, returns_col="returns")
)

In [95]:
df_rolling_historical_var

idx,portfolio_id,date,value,returns,vol,rolled_returns,var
u32,str,date,"decimal[*,20]",f64,f64,list[f64],f64
0,"""0b209d5c-12ed-48af-a597-b1e22f…",2018-09-05,39717.53916860000000000000,-0.014411,null,[-0.014411],null
1,"""0b209d5c-12ed-48af-a597-b1e22f…",2018-09-06,39755.61926370000000000000,0.000959,null,"[-0.014411, 0.000959]",null
2,"""0b209d5c-12ed-48af-a597-b1e22f…",2018-09-07,39555.69876450000000000000,-0.005029,null,"[-0.014411, 0.000959, -0.005029]",null
3,"""0b209d5c-12ed-48af-a597-b1e22f…",2018-09-10,39165.37778990000000000000,-0.009868,null,"[-0.014411, 0.000959, … -0.009868]",null
4,"""0b209d5c-12ed-48af-a597-b1e22f…",2018-09-11,39251.05800390000000000000,0.002188,null,"[-0.014411, 0.000959, … 0.002188]",null
…,…,…,…,…,…,…,…
1747,"""925a0ff5-3258-44c4-86eb-3de00c…",2025-08-22,22570.00000000000000000000,0.004898,0.007212,"[0.016548, -0.002791, … 0.004898]",-0.029006
1748,"""925a0ff5-3258-44c4-86eb-3de00c…",2025-08-25,22710.00000000000000000000,0.006203,0.006735,"[-0.002791, 0.003731, … 0.006203]",-0.029006
1749,"""925a0ff5-3258-44c4-86eb-3de00c…",2025-08-26,22520.00000000000000000000,-0.008366,0.006272,"[0.003731, 0.0, … -0.008366]",-0.029006


In [72]:
df_exante_perf

portfolio_id,date,value,returns
str,date,"decimal[*,20]",f64
"""78e37b84-0a65-4f70-a706-ae0175…",2018-09-05,87373.96029150000000000000,-0.001001
"""78e37b84-0a65-4f70-a706-ae0175…",2018-09-06,86722.35128220000000000000,-0.007458
"""78e37b84-0a65-4f70-a706-ae0175…",2018-09-07,86527.84113020000000000000,-0.002243
"""78e37b84-0a65-4f70-a706-ae0175…",2018-09-10,86625.09620620000000000000,0.001124
"""78e37b84-0a65-4f70-a706-ae0175…",2018-09-11,86654.27272900000000000000,0.000337
…,…,…,…
"""16cc08b3-c767-464d-8d9c-573d4d…",2025-08-22,648500.00000000000000000000,0.015344
"""16cc08b3-c767-464d-8d9c-573d4d…",2025-08-25,645640.00000000000000000000,-0.00441
"""16cc08b3-c767-464d-8d9c-573d4d…",2025-08-26,648260.00000000000000000000,0.004058


In [ ]:
df_rolling_vol = ()

In [60]:
df_market_data

instrument_id,date,price
str,date,"decimal[*,10]"
"""3c7f3c30-4848-48ce-b2a5-3fc1b9…",2025-08-28,37.6700000000
"""6c125ed4-4a90-41c4-b389-226eb7…",2025-08-28,87.2200000000
"""491f4737-7acc-4f1a-b486-e6cc01…",2025-08-28,111.2900000000
"""7edd1534-ecb5-4a73-a3b0-4bcfe6…",2025-08-28,73.0900000000
"""07602a17-b596-47d1-adc2-bda6d3…",2025-08-28,72.7300000000
…,…,…
"""7edd1534-ecb5-4a73-a3b0-4bcfe6…",2018-11-05,42.0968737433
"""c6d94933-a248-4df5-a56e-b2c0c0…",2018-11-05,102.3831230164
"""5817580d-febb-47d3-a1dd-70cae7…",2018-11-05,22.8190594604


In [58]:
df_market_data

instrument_id,date,data_type,value
object,date,str,"decimal[*,10]"
3c7f3c30-4848-48ce-b2a5-3fc1b9e31b1e,2025-08-28,"""adj_close""",37.6700000000
6c125ed4-4a90-41c4-b389-226eb7e93b07,2025-08-28,"""adj_close""",87.2200000000
491f4737-7acc-4f1a-b486-e6cc014c5e76,2025-08-28,"""adj_close""",111.2900000000
7edd1534-ecb5-4a73-a3b0-4bcfe6edc0e1,2025-08-28,"""adj_close""",73.0900000000
07602a17-b596-47d1-adc2-bda6d3cda617,2025-08-28,"""adj_close""",72.7300000000
…,…,…,…
e24878f5-b374-4d80-ba37-234362ea1ec9,2018-09-04,"""adj_close""",44.1677008316
41712141-41c4-4e22-83d1-8cc45c44fff1,2018-09-04,"""adj_close""",40.2982606186
929e3e08-574e-4b80-ab1d-57bdef534444,2018-09-04,"""adj_close""",44.0783497738


In [ ]:
df_positions

date,portfolio_id,instrument_id,quantity
date,object,object,"decimal[*,8]"
2025-08-29,e3a50d76-dcb7-4377-8ad3-8ff9f442752d,fb39c249-6cb1-47b6-9220-6c0f8daac13a,1000.00000000
2025-08-29,4daf79fe-feb9-4c5a-ba1f-a67b965c679a,6c125ed4-4a90-41c4-b389-226eb7e93b07,1000.00000000
2025-08-29,d7da851b-20c5-4de0-9c93-68a9f16fb55e,02f2fb63-27d1-43bf-953d-7249c482b082,1000.00000000
2025-08-29,0b209d5c-12ed-48af-a597-b1e22f7780b6,41712141-41c4-4e22-83d1-8cc45c44fff1,1000.00000000
2025-08-29,f4dd8467-36cf-4d10-93d3-a0699ae8eb70,c135cb6f-2a23-4032-ae80-856a310752a8,1000.00000000
…,…,…,…
2025-08-29,f0085a41-9c45-4bd5-8a9b-456a3115f72b,727b0b7c-268d-42c4-bb48-11020f53958a,1000.00000000
2025-08-29,ca366c28-aff3-45f6-ae3b-37079cdc445c,6444a8ec-8304-4918-a36d-0766727c57d7,1000.00000000
2025-08-29,67756108-d9ee-44e0-ac13-0e133d3e39c0,07602a17-b596-47d1-adc2-bda6d3cda617,1000.00000000


In [19]:
holdings

In [13]:
portfolios_dict

{UUID('bd3425c7-1dea-44bd-b345-d7517316f314'): {'portfolio': <worker.database.Portfolio at 0x11780fa10>,
  'holdings': [<worker.database.Holding at 0x116f56040>]},
 UUID('30ecbfb2-2fa9-4611-afcf-8b8fd61aa429'): {'portfolio': <worker.database.Portfolio at 0x11780fa80>,
  'holdings': [<worker.database.Holding at 0x116f54750>]},
 UUID('7509fde2-c757-486d-9b53-90f28b5d5a0c'): {'portfolio': <worker.database.Portfolio at 0x11780faf0>,
  'holdings': [<worker.database.Holding at 0x116f546e0>]},
 UUID('6c5d5b6e-f50a-40fe-ba5e-b208e1ecb458'): {'portfolio': <worker.database.Portfolio at 0x11780cde0>,
  'holdings': [<worker.database.Holding at 0x116f55cc0>]},
 UUID('24d4e3a0-5ea6-4ace-b18a-8ac9a16fcb8e'): {'portfolio': <worker.database.Portfolio at 0x11780dbe0>,
  'holdings': [<worker.database.Holding at 0x116f54670>]},
 UUID('2b8e2f4b-c492-4521-89f3-0a4f8a2afcb1'): {'portfolio': <worker.database.Portfolio at 0x11780d240>,
  'holdings': [<worker.database.Holding at 0x116f55c50>]},
 UUID('6528a25b-

In [7]:
portfolios_dict

{UUID('bd3425c7-1dea-44bd-b345-d7517316f314'): {'portfolio': <worker.database.Portfolio at 0x116f11010>,
  'holdings': [<worker.database.Holding at 0x1169875b0>]},
 UUID('30ecbfb2-2fa9-4611-afcf-8b8fd61aa429'): {'portfolio': <worker.database.Portfolio at 0x116eede50>,
  'holdings': [<worker.database.Holding at 0x116f04c20>]},
 UUID('7509fde2-c757-486d-9b53-90f28b5d5a0c'): {'portfolio': <worker.database.Portfolio at 0x116eedd10>,
  'holdings': [<worker.database.Holding at 0x116f052b0>]},
 UUID('6c5d5b6e-f50a-40fe-ba5e-b208e1ecb458'): {'portfolio': <worker.database.Portfolio at 0x116e7a520>,
  'holdings': [<worker.database.Holding at 0x116f05240>]},
 UUID('24d4e3a0-5ea6-4ace-b18a-8ac9a16fcb8e'): {'portfolio': <worker.database.Portfolio at 0x116e7a2c0>,
  'holdings': [<worker.database.Holding at 0x116f05320>]},
 UUID('2b8e2f4b-c492-4521-89f3-0a4f8a2afcb1'): {'portfolio': <worker.database.Portfolio at 0x116f34cb0>,
  'holdings': [<worker.database.Holding at 0x116f05390>]},
 UUID('6528a25b-